# PG Primary + Private ES Index

**Routing pattern:**
- WRITE: `[postgresql, elasticsearch_private]` — PG stores full items; private ES index stores geoid tokens only
- READ: `[postgresql]` — browsing via PG
- SEARCH: `[elasticsearch_private]` — spatial discovery via geoid-only private index

Use this pattern for location-aware discovery that keeps item attributes private — the ES private index stores only spatial tokens; full item data stays in PostgreSQL.

In [ ]:
import httpx

import os
BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:80"
CATALOG_ID = "demo-private"
COLLECTION_ID = "protected-parcels"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

print("Client ready")

## Step 1 — Create catalog and collection

In [ ]:
import asyncio

r = await client.post(f"/stac/catalogs", json={"id": CATALOG_ID, "title": "Demo Obfuscated", "description": "Obfuscated ES demo"})
_check(r)
# Wait for provisioning to complete before creating collections
for _ in range(30):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("Warning: catalog still provisioning after 30s")

In [ ]:
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Protected Parcels",
    "description": "Land parcels — attributes anonymized in search",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r)

## Step 2 — Configure routing

In [ ]:
routing = {
    "operations": {
        "WRITE": [
            {"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal", "write_mode": "sync"},
            {"driver_id": "ItemsElasticsearchObfuscatedDriver", "on_failure": "warn", "write_mode": "async"}
        ],
        "READ": [{"driver_id": "ItemsPostgresqlDriver"}],
        "SEARCH": [{"driver_id": "ItemsElasticsearchObfuscatedDriver"}],
        "METADATA": [{"driver_id": "ItemsPostgresqlDriver"}]
    }
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig",
    json=routing,
)
_check(r)

## Step 3 — Insert test items

In [ ]:
import json

features = [
    {
        "type": "Feature",
        "id": f"parcel-{i:03d}",
        "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.01, 41.9 + i * 0.01]},
        "bbox": [12.5 + i * 0.01, 41.9 + i * 0.01, 12.5 + i * 0.01, 41.9 + i * 0.01],
        "properties": {"owner": f"Owner {i}", "area_m2": 500 + i * 10, "cadastral_ref": f"IT{i:05d}"}
    }
    for i in range(5)
]

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features},
)
_check(r, f"{len(features)} items sent")

## Step 4 — Read items from PG

In [ ]:
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=10")
data = r.json()
print(f"Items returned: {data.get('numberReturned', len(data.get('features', [])))}")
for f in data.get("features", []):
    print(" ", f["id"], f["properties"].get("owner"))

## Step 5 — Search via Private ES Index (geoid tokens only)

In [ ]:
r = await client.post(
    f"/search/catalogs/{CATALOG_ID}/items-search",
    json={"collections": [COLLECTION_ID], "limit": 10},
)
data = r.json()
print(f"Search results: {len(data.get('features', []))}")
for f in data.get("features", []):
    # Private ES index: only geoid tokens returned, no item attributes
    print(" ", f.get("id"), "properties:", list(f.get("properties", {}).keys()))

In [ ]:
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
_check(r, "Cleanup")
await client.aclose()